# Counterfactual Fairness Evaluation v3
**KLUE-RoBERTa 기반 한국어 혐오 탐지 모델의 counterfactual / hard-case 평가**

**사전 준비:**
1. 런타임 -> 런타임 유형 변경 -> **T4 GPU** 선택
2. Google Drive의 원하는 폴더에 `kold_v1.json`, `counterfactual_eval_v3.json` 업로드
3. 아래 `DRIVE_DIR` 경로를 해당 폴더로 수정 후 실행


## 0. 패키지 설치

In [ ]:
!pip install -q transformers

## 1. Google Drive 마운트 & 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── 여기만 수정 ─────────────────────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/nlp_project'  # 데이터 파일이 있는 Drive 폴더
# ──────────────────────────────────────────────────────────

import os
KOLD_PATH   = os.path.join(DRIVE_DIR, 'kold_v1.json')
FLIP_PATH   = os.path.join(DRIVE_DIR, 'counterfactual_eval_v3.json')
CKPT_DIR    = os.path.join(DRIVE_DIR, 'checkpoints')
RESULT_PATH = os.path.join(DRIVE_DIR, 'counterfactual_eval_v3_results.json')

os.makedirs(CKPT_DIR, exist_ok=True)

print(f'KOLD     : {KOLD_PATH}  exists={os.path.exists(KOLD_PATH)}')
print(f'EvalSet  : {FLIP_PATH}  exists={os.path.exists(FLIP_PATH)}')
print(f'Ckpt dir : {CKPT_DIR}')


## 2. 설정

In [ ]:
SUBSET     = 400    # KOLD 사용 샘플 수 (0 = 전체)
EPOCHS     = 5
BATCH_SIZE = 16     # T4 GPU 기준
MAX_LEN    = 128
HEAD_DIM   = 128
LR         = 2e-5
LAMBDA     = 0.05
SEED       = 42
MODEL_NAME = 'klue/roberta-base'

## 3. 모델 정의

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


class BaselineClassifier(nn.Module):
    def __init__(self, model_name='klue/roberta-base', dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.classifier(self.dropout(cls)).squeeze(1), None, None


class DualHeadClassifier(nn.Module):
    def __init__(self, model_name='klue/roberta-base', head_dim=128, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.cls_head = nn.Sequential(nn.Linear(hidden, head_dim), nn.ReLU())
        self.sem_head = nn.Sequential(nn.Linear(hidden, head_dim), nn.ReLU())
        self.hate_classifier = nn.Linear(head_dim, 1)

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        h = self.dropout(cls)
        z_cls = self.cls_head(h)
        z_sem = self.sem_head(h)
        return self.hate_classifier(z_cls).squeeze(1), z_cls, z_sem


def orthogonality_loss(z_cls, z_sem):
    return (F.cosine_similarity(z_cls, z_sem, dim=1) ** 2).mean()

## 4. 데이터 로드

In [ ]:
import json
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

random.seed(SEED)


def load_kold(path, subset, seed=42):
    with open(path, encoding='utf-8') as f:
        raw = json.load(f)
    examples = [(d['comment'], int(d['OFF'])) for d in raw if d['comment'].strip()]
    if subset:
        rng = random.Random(seed)
        pos = [e for e in examples if e[1] == 1]
        neg = [e for e in examples if e[1] == 0]
        rng.shuffle(pos); rng.shuffle(neg)
        examples = pos[:subset//2] + neg[:subset//2]
        rng.shuffle(examples)
    return examples


class KOLDDataset(Dataset):
    def __init__(self, examples, tokenizer, max_len):
        self.examples, self.tokenizer, self.max_len = examples, tokenizer, max_len

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        text, label = self.examples[idx]
        enc = self.tokenizer(text, max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float32),
        }


all_data = load_kold(KOLD_PATH, SUBSET, seed=SEED)
random.Random(SEED).shuffle(all_data)
cut = int(len(all_data) * 0.8)
train_data, val_data = all_data[:cut], all_data[cut:]
print(f'KOLD: total={len(all_data)}  train={len(train_data)}  val={len(val_data)}')

tokenizer    = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(KOLDDataset(train_data, tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(KOLDDataset(val_data,   tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

with open(FLIP_PATH, encoding='utf-8') as f:
    flip_data = json.load(f)
pairs = flip_data['pairs']
category_meta = flip_data['meta'].get('categories', {})
print(f"Eval v{flip_data['meta'].get('version', '?')}: {len(pairs)} pairs")
for cat, meta in category_meta.items():
    print(f"  - {cat}: {meta['count']} pairs  label={meta['expected_label']}")


## 5. 훈련 & 평가 함수

In [ ]:
import gc
from collections import defaultdict, Counter


def train_epoch(model, loader, optimizer):
    model.train()
    criterion = nn.BCEWithLogitsLoss()
    total_loss = total_ortho = n = 0
    for batch in loader:
        ids, mask, y = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        logits, z_cls, z_sem = model(ids, mask)
        cls_l = criterion(logits, y)
        o_l = orthogonality_loss(z_cls, z_sem) if z_cls is not None else torch.tensor(0.0, device=device)
        loss = cls_l + LAMBDA * o_l
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs = ids.size(0)
        total_loss  += loss.item() * bs
        total_ortho += o_l.item() * bs
        n += bs
    return total_loss / max(n,1), total_ortho / max(n,1)


def val_eval(model):
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss = correct = total = tp = fp = fn = tn = 0
    with torch.no_grad():
        for batch in val_loader:
            ids, mask, y = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
            logits, _, _ = model(ids, mask)
            total_loss += criterion(logits, y).item() * ids.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == y).sum().item(); total += ids.size(0)
            tp += ((preds==1)&(y==1)).sum().item(); fp += ((preds==1)&(y==0)).sum().item()
            fn += ((preds==0)&(y==1)).sum().item(); tn += ((preds==0)&(y==0)).sum().item()
    prec = tp/max(tp+fp,1); rec = tp/max(tp+fn,1)
    f1   = 2*prec*rec/max(prec+rec,1e-8)
    f1n  = 2*(tn/max(tn+fn,1))*(tn/max(tn+fp,1))/max((tn/max(tn+fn,1))+(tn/max(tn+fp,1)),1e-8)
    return {'loss': total_loss/max(total,1), 'acc': correct/max(total,1),
            'macro_f1': (f1+f1n)/2, 'tp':tp,'fp':fp,'fn':fn,'tn':tn}


def train_model(name, model, ckpt_name):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    best_f1 = 0.0
    ckpt_path = os.path.join(CKPT_DIR, ckpt_name)

    print(f"
{'='*60}
  {name}
  epochs={EPOCHS}  lr={LR}  lambda={LAMBDA}
{'='*60}")
    print(f"{'Ep':>3}  {'TrLoss':>8}  {'OrthoL':>8}  {'ValLoss':>8}  {'MacroF1':>8}  {'Acc':>6}")

    for epoch in range(1, EPOCHS + 1):
        tr_loss, ortho_l = train_epoch(model, train_loader, optimizer)
        m = val_eval(model)
        print(f"{epoch:>3}  {tr_loss:>8.4f}  {ortho_l:>8.4f}  {m['loss']:>8.4f}  {m['macro_f1']:>8.3f}  {m['acc']:>6.3f}", end='')

        if m['macro_f1'] >= best_f1:
            best_f1 = m['macro_f1']
            torch.save(model.state_dict(), ckpt_path)
            print('  * saved', end='')
        print()

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    final = val_eval(model)
    print(f"  Best macro_F1={best_f1:.4f}  (checkpoint: {ckpt_path})")
    print(f"  Confusion: tp={final['tp']} fp={final['fp']} fn={final['fn']} tn={final['tn']}")
    return model, final


def predict(model, text):
    model.eval()
    with torch.no_grad():
        enc = tokenizer(text, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
        logits, _, _ = model(enc['input_ids'].to(device), enc['attention_mask'].to(device))
        prob = torch.sigmoid(logits).item()
    return int(prob >= 0.5), round(prob, 3)


def eval_counterfactual_set(model, model_name):
    overall_consistent = 0
    overall_correct = 0
    total_preds = 0
    by_cat = defaultdict(lambda: {'total': 0, 'consistent': 0, 'correct': 0, 'flips': []})

    for p in pairs:
        op, oprob = predict(model, p['original'])
        cp, cprob = predict(model, p['counterfactual'])
        expected = p['expected_label']
        cat = p.get('category', 'uncategorized')
        consistent = (op == cp)
        orig_correct = (op == expected)
        cf_correct = (cp == expected)

        overall_consistent += int(consistent)
        overall_correct += int(orig_correct) + int(cf_correct)
        total_preds += 2

        by_cat[cat]['total'] += 1
        by_cat[cat]['consistent'] += int(consistent)
        by_cat[cat]['correct'] += int(orig_correct) + int(cf_correct)

        if not consistent or not (orig_correct and cf_correct):
            by_cat[cat]['flips'].append({
                **p,
                'orig_pred': op, 'orig_prob': oprob,
                'cf_pred': cp, 'cf_prob': cprob,
                'orig_correct': orig_correct,
                'cf_correct': cf_correct,
            })

    total = len(pairs)
    overall = {
        'consistency': overall_consistent / max(total, 1),
        'pair_accuracy': overall_correct / max(total_preds, 1),
        'consistent': overall_consistent,
        'total': total,
    }

    print(f"
  [{model_name}] overall consistency={overall['consistency']:.2f}  pair_acc={overall['pair_accuracy']:.2f}")
    print('  Category breakdown:')
    for cat, stats in by_cat.items():
        meta = category_meta.get(cat, {})
        exp = meta.get('expected_label', '?')
        cons = stats['consistent'] / max(stats['total'], 1)
        acc = stats['correct'] / max(stats['total'] * 2, 1)
        print(f"    {cat:<24} n={stats['total']:>2}  y={exp}  consistency={cons:.2f}  pair_acc={acc:.2f}")
        for row in stats['flips'][:3]:
            print(f"      [CASE] id={row['id']:>2}  {row['identity_orig']} -> {row['identity_cf']}  orig={row['orig_pred']}({row['orig_prob']:.2f}) cf={row['cf_pred']}({row['cf_prob']:.2f})")
            print(f"             orig: {row['original'][:80]}")
            print(f"             cf  : {row['counterfactual'][:80]}")
        if len(stats['flips']) > 3:
            print(f"             ... and {len(stats['flips']) - 3} more")

    overall['by_category'] = {
        cat: {
            'total': stats['total'],
            'consistency': stats['consistent'] / max(stats['total'], 1),
            'pair_accuracy': stats['correct'] / max(stats['total'] * 2, 1),
            'cases': stats['flips'],
        }
        for cat, stats in by_cat.items()
    }
    return overall


print('함수 정의 완료')


## 6. Baseline 훈련 + 평가

In [ ]:
torch.manual_seed(SEED)
baseline = BaselineClassifier(MODEL_NAME).to(device)
baseline, base_final = train_model('Baseline (Single-head)', baseline, 'baseline_best.pt')

print("
" + "="*60 + "
  COUNTERFACTUAL / HARD-CASE EVALUATION
" + "="*60)
res_base = eval_counterfactual_set(baseline, 'Baseline')

del baseline; gc.collect(); torch.cuda.empty_cache()


## 7. Dual-head 훈련 + 평가

In [ ]:
torch.manual_seed(SEED)
dual = DualHeadClassifier(MODEL_NAME, HEAD_DIM).to(device)
dual, dual_final = train_model(f'Dual-head + Ortho (lambda={LAMBDA})', dual, 'dual_best.pt')

res_dual = eval_counterfactual_set(dual, 'Dual-head')

del dual; gc.collect(); torch.cuda.empty_cache()


## 8. 최종 결과 요약 & 저장

In [ ]:
# checkpoint에서 baseline 로드 (재훈련 불필요)
baseline_diag = BaselineClassifier(MODEL_NAME).to(device)
baseline_diag.load_state_dict(torch.load(os.path.join(CKPT_DIR, 'baseline_best.pt'), map_location=device))
baseline_diag.eval()

print('[ Baseline - category별 패턴 ]')
for cat in category_meta.keys():
    cat_pairs = [p for p in pairs if p.get('category') == cat]
    counts = {'both_1': 0, 'both_0': 0, 'flip_1_to_0': 0, 'flip_0_to_1': 0}
    print("
" + '-' * 70)
    print(f"{cat}  (n={len(cat_pairs)}, expected_label={category_meta[cat]['expected_label']})")
    for p in cat_pairs:
        op, oprob = predict(baseline_diag, p['original'])
        cp, cprob = predict(baseline_diag, p['counterfactual'])
        if   op == 1 and cp == 1: counts['both_1'] += 1
        elif op == 0 and cp == 0: counts['both_0'] += 1
        elif op == 1 and cp == 0: counts['flip_1_to_0'] += 1
        else:                     counts['flip_0_to_1'] += 1
    total = max(len(cat_pairs), 1)
    for k, v in counts.items():
        print(f"  {k:<12} {v:>2}/{total}  ({v/total:.2f})")

del baseline_diag; gc.collect(); torch.cuda.empty_cache()


## 8. Diagnostic: consistent 쌍 전체 예측 분포

저장된 checkpoint로 재훈련 없이 바로 실행 가능.  
**"둘 다 1"인 쌍이 대부분이면 → structural framing shortcut 증거.**

In [ ]:
delta = res_dual['consistency'] - res_base['consistency']
acc_delta = res_dual['pair_accuracy'] - res_base['pair_accuracy']

print('='*60 + '
  SUMMARY
' + '='*60)
print(f"
  KOLD val:")
print(f"    Baseline  macro_F1={base_final['macro_f1']:.3f}  acc={base_final['acc']:.3f}")
print(f"    Dual-head macro_F1={dual_final['macro_f1']:.3f}  acc={dual_final['acc']:.3f}")
print(f"
  Overall counterfactual metrics:")
print(f"    Baseline  consistency={res_base['consistency']:.2f}  pair_acc={res_base['pair_accuracy']:.2f}")
print(f"    Dual-head consistency={res_dual['consistency']:.2f}  pair_acc={res_dual['pair_accuracy']:.2f}")
print(f"    Delta     consistency={delta:+.2f}  pair_acc={acc_delta:+.2f}")

print(f"
  Per-category comparison:")
for cat in category_meta.keys():
    b = res_base['by_category'].get(cat, {})
    d = res_dual['by_category'].get(cat, {})
    print(f"    {cat:<24}  Base(cons={b.get('consistency', 0):.2f}, acc={b.get('pair_accuracy', 0):.2f})  Dual(cons={d.get('consistency', 0):.2f}, acc={d.get('pair_accuracy', 0):.2f})")

output = {
    'config': {'model': MODEL_NAME, 'subset': SUBSET, 'epochs': EPOCHS,
               'batch_size': BATCH_SIZE, 'max_len': MAX_LEN, 'lr': LR, 'lambda_ortho': LAMBDA},
    'dataset_meta': flip_data['meta'],
    'kold_val': {
        'baseline':  {'macro_f1': base_final['macro_f1'], 'acc': base_final['acc']},
        'dual_head': {'macro_f1': dual_final['macro_f1'], 'acc': dual_final['acc']}},
    'counterfactual_eval': {
        'baseline': res_base,
        'dual_head': res_dual,
        'consistency_delta': delta,
        'pair_accuracy_delta': acc_delta,
    }
}
with open(RESULT_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f"
  결과 저장: {RESULT_PATH}")
